# BBO overview

Initial Polars exploration of the canonical Parquet capture. The notebook
keeps the large order-book-level table out of memory.

In [5]:
from pathlib import Path
import polars as pl

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / 'backend').exists() else cwd.parents[1]
DATASET = PROJECT_ROOT / 'experiments' / 'data' / 'parquet' / 'btc-capture-001'

if not DATASET.exists():
    raise FileNotFoundError(f'Copy the dataset to {DATASET}')

bbo_files = list(
    DATASET.glob("**/event_type=best_bid_offers/**/*.parquet")
)

print(len(bbo_files))

19


In [8]:
bbo = (
    pl.scan_parquet(
        [str(path) for path in bbo_files],
        hive_partitioning=True,
    )
    .select([
        "capture_sequence",
        "venue",
        "market_coin",
        "local_receive_time",
        "best_bid_price",
        "best_bid_quantity",
        "best_ask_price",
        "best_ask_quantity",
    ])
    .rename({
        "capture_sequence": "sequence",
        "market_coin": "market",
        "local_receive_time": "local_receive",
        "best_bid_price": "bid_price",
        "best_bid_quantity": "bid_quantity",
        "best_ask_price": "ask_price",
        "best_ask_quantity": "ask_quantity",
    })
)

bbo.group_by("venue").len().collect()

venue,len
str,u32
"""hyperliquid""",11755


In [9]:
bbo.collect().head()

sequence,venue,market,local_receive,bid_price,bid_quantity,ask_price,ask_quantity
u64,str,str,u64,"decimal[38,18]","decimal[38,18]","decimal[38,18]","decimal[38,18]"
57352,"""hyperliquid""","""BTC""",1369193854076,80886.000000000000000000,6.161480000000000000,80887.000000000000000000,3.349320000000000000
57354,"""hyperliquid""","""BTC""",1369226942539,80886.000000000000000000,6.134210000000000000,80887.000000000000000000,3.341910000000000000
57362,"""hyperliquid""","""BTC""",1369441749826,80886.000000000000000000,5.738180000000000000,80887.000000000000000000,3.349320000000000000
57366,"""hyperliquid""","""BTC""",1369498962993,80886.000000000000000000,5.428650000000000000,80887.000000000000000000,3.349320000000000000
57381,"""hyperliquid""","""BTC""",1369933034813,80886.000000000000000000,5.738180000000000000,80887.000000000000000000,3.349320000000000000
